# LoRA Training for Intention Identification - Qwen3-4B-Instruct with Flash-attention

## 1. Preparation

### 1.1 Import dependencies

In [1]:
import os
import gc
from datetime import datetime
import json
import ast
import re
from tqdm import tqdm
import torch
from datasets import load_dataset, DatasetDict
from peft import PeftModel

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    EarlyStoppingCallback,
    Trainer,
    set_seed
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training
)

In [2]:
import warnings
warnings.filterwarnings('ignore')

### 1.2 Set constants

In [3]:
MODEL_NAME = "../models/Qwen/Qwen3-4B-Instruct-2507"
DATA_PATH = "../data/final/LoRA_samples_sum_fixed.jsonl"

OUTPUT_DIR = "./fine_tuning_output"

MAX_LENGTH = 2048
SEED = 42

set_seed(SEED)

Enable TensorFloat-32 (TF32) on CUDA and cuDNN to accelerate matrix and convolution operations with minimal precision loss.

In [4]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

## 2. Dataset

### 2.1 Load and shuffle the dataset

In [5]:
# Load dataset
ds_raw = load_dataset("json", data_files=DATA_PATH)["train"]

Generating train split: 0 examples [00:00, ? examples/s]

In [6]:
print(ds_raw)

Dataset({
    features: ['messages'],
    num_rows: 10104
})


In [7]:
# Shuffle the datasets
shuffled_ds = ds_raw.shuffle(seed=SEED)

Check the shuffled dataset.

In [8]:
shuffled_ds[:3]

{'messages': [[{'role': 'user',
    'content': "## === 你的角色描述和背景信息（仅供参考） ===\n你是一个家装博览会的电话邀约专员，负责邀请南通市民参加本周末的家装建材采购节。\n\n## === 你的核心任务（必须遵守） ===\n你需要根据以下指示判断**最后一次用户输入**的意图，然后**仅输出一个JSON对象**，绝对不能是其他任何格式\n\n### 重要规则：\n1. 你只负责判断意图，**不进行对话**\n2. 无论对话历史如何，你的**输出必须是且仅是一个JSON对象**，不得输出为空、不得输出其他格式的数据\n3. 禁止输出任何自然语言、解释或额外内容\n4. 如果之前的回复是自然语言，忽略它并继续输出JSON\n5. JSON必须包含且仅包含两个字段：`input_summary`、`intention_id`。字段值必须为字符串，格式必须严格遵循：{'input_summary': '...', 'intention_id': '...'}\n\n### JSON字段说明：\n1. **input_summary**\n- 务必参考**智能助手和用户的对话历史**，用不超过10个汉字，清晰简洁地概括**最后一次用户输入**的核心内容\n- 示例：“用户有兴趣参加活动”、“用户不想被打扰”、“用户询问你是谁”\n\n2. **intention_id**\n- 务必参考**智能助手和用户的对话历史**，判断**最后一次用户输入**的意图\n- 仅可从以下【意图库列表】或【知识库列表】中，根据**意图名称**和**意图说明**，选择**唯一最匹配**的意图\n- 若最后一次用户输入无法匹配任一意图，输出 `intention_id` 为 'others'\n- 如果最后一次用户输入的语义匹配某一条意图的**意图名称**和**意图说明**，则将这条意图的**意图id**输出为 `intention_id`\n- 禁止自行创建或修改 `intention_id`，其值必须严格来自两个列表中的意图id，或为 'others'\n- 选择优先级说明：\n  - 如果两个列表均有内容，则同时使用【知识库列表】和【意图库列表】判别用户的意图，不分先后，当匹配成功，将**意图id**输出为 `intenti

Under a dict, there is one key "messages", the value is a list. Each item in the list is a data sample (also a list).

### 2.2 Split the dataset
Split it into train (75%), validation (15%), test (10%)

In [9]:
train_temp = shuffled_ds.train_test_split(test_size=0.25, seed=SEED)

val_test = train_temp['test'].train_test_split(test_size=0.4, seed=42)

ds = DatasetDict({
    'train': train_temp['train'], # 75%
    'validation': val_test['train'], # 15%
    'test': val_test['test'] # 10%
})

Check the split datasets.

In [10]:
ds["train"]

Dataset({
    features: ['messages'],
    num_rows: 7578
})

In [11]:
ds["validation"]

Dataset({
    features: ['messages'],
    num_rows: 1515
})

In [12]:
ds["test"]

Dataset({
    features: ['messages'],
    num_rows: 1011
})

In [13]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 7578
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 1515
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 1011
    })
})


### 2.3 Tokenization

Load the tokenizer.

In [14]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

Configure and check the tokenizer. In training, we set the padding on the right to keep the labels aligned, and make it easy to mask loss.

In [15]:
tokenizer.padding_side = "right"
tokenizer.truncation_side = "left"

In [16]:
tokenizer

Qwen2Tokenizer(name_or_path='../models/Qwen/Qwen3-4B-Instruct-2507', vocab_size=151643, model_max_length=1010000, padding_side='right', truncation_side='left', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151649: AddedToken("<|box_end|>"

In [17]:
# The tokenizer already has PAD token and EOS token. There is no need for extra set up.
print(f"\\nTokenizer info:")
print(f"PAD token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")

\nTokenizer info:
PAD token: <|endoftext|> (ID: 151643)
EOS token: <|im_end|> (ID: 151645)


Set up the process function for correct tokenization for the data samples.

In [18]:
def process_func(example):

    messages = example["messages"]

    instruction_text = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True
    )

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    ) + tokenizer.eos_token

    tokenized_full = tokenizer(
        full_text,
        max_length=MAX_LENGTH,
        truncation=True,
        add_special_tokens=False
    )

    tokenized_instruction = tokenizer(
        instruction_text,
        max_length=MAX_LENGTH,
        truncation=True,
        add_special_tokens=False
    )

    input_ids = tokenized_full["input_ids"]
    attention_mask = tokenized_full["attention_mask"]

    instruction_len = len(tokenized_instruction["input_ids"])

    labels = [-100] * instruction_len + input_ids[instruction_len:]
    labels = labels[:len(input_ids)]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

Tokenize, and check the tokenized data.

In [19]:
tokenized_ds = ds.map(
    process_func,
    remove_columns=ds["train"].column_names,
    num_proc=os.cpu_count(),
    desc="Processing dataset"
)

Processing dataset (num_proc=128):   0%|          | 0/7578 [00:00<?, ? examples/s]

Processing dataset (num_proc=128):   0%|          | 0/1515 [00:00<?, ? examples/s]

Processing dataset (num_proc=128):   0%|          | 0/1011 [00:00<?, ? examples/s]

In [20]:
tokenized_ds

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 7578
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1515
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1011
    })
})

In [21]:
len(tokenized_ds["train"]["input_ids"][5])

1428

In [22]:
# Check sequence lengths
max_len = max(len(x) for x in tokenized_ds["train"]["input_ids"])
avg_len = sum(len(x) for x in tokenized_ds["train"]["input_ids"]) / len(tokenized_ds["train"])
print(f"Sequence length - Max: {max_len}, Avg: {avg_len:.1f}")

Sequence length - Max: 2043, Avg: 1499.5


In [23]:
tokenized_ds["train"][2]

{'input_ids': [151644,
  872,
  198,
  565,
  2049,
  220,
  103929,
  100780,
  53481,
  33108,
  102193,
  27369,
  9909,
  110079,
  7552,
  2049,
  198,
  105043,
  102192,
  106668,
  9370,
  105041,
  3837,
  85106,
  72064,
  104235,
  20002,
  81167,
  64471,
  110148,
  90395,
  99553,
  105674,
  89656,
  3407,
  565,
  2049,
  220,
  103929,
  100185,
  88802,
  9909,
  100645,
  105389,
  7552,
  2049,
  198,
  112735,
  100345,
  87752,
  102857,
  104317,
  334,
  117241,
  20002,
  31196,
  334,
  9370,
  111450,
  3837,
  101889,
  334,
  99373,
  66017,
  46944,
  5370,
  64429,
  334,
  3837,
  101244,
  53153,
  20412,
  92894,
  99885,
  68805,
  271,
  14374,
  220,
  99335,
  104190,
  28311,
  16,
  13,
  220,
  56568,
  91680,
  100668,
  104317,
  111450,
  3837,
  334,
  16530,
  71817,
  105051,
  1019,
  17,
  13,
  220,
  100783,
  105051,
  100022,
  100007,
  3837,
  103929,
  334,
  66017,
  100645,
  20412,
  100136,
  99373,
  101909,
  5370,
  64429,


In [24]:
tokenizer.decode(tokenized_ds["train"][2]["input_ids"])

'<|im_start|>user\n## === 你的角色描述和背景信息（仅供参考） ===\n你是家居博览会的客服，需要联系报名用户确认是否参会，并提供展会详情。\n\n## === 你的核心任务（必须遵守） ===\n你需要根据以下指示判断**最后一次用户输入**的意图，然后**仅输出一个JSON对象**，绝对不能是其他任何格式\n\n### 重要规则：\n1. 你只负责判断意图，**不进行对话**\n2. 无论对话历史如何，你的**输出必须是且仅是一个JSON对象**，不得输出为空、不得输出其他格式的数据\n3. 禁止输出任何自然语言、解释或额外内容\n4. 如果之前的回复是自然语言，忽略它并继续输出JSON\n5. JSON必须包含且仅包含两个字段：`input_summary`、`intention_id`。字段值必须为字符串，格式必须严格遵循：{\'input_summary\': \'...\', \'intention_id\': \'...\'}\n\n### JSON字段说明：\n1. **input_summary**\n- 务必参考**智能助手和用户的对话历史**，用不超过10个汉字，清晰简洁地概括**最后一次用户输入**的核心内容\n- 示例：“用户有兴趣参加活动”、“用户不想被打扰”、“用户询问你是谁”\n\n2. **intention_id**\n- 务必参考**智能助手和用户的对话历史**，判断**最后一次用户输入**的意图\n- 仅可从以下【意图库列表】或【知识库列表】中，根据**意图名称**和**意图说明**，选择**唯一最匹配**的意图\n- 若最后一次用户输入无法匹配任一意图，输出 `intention_id` 为 \'others\'\n- 如果最后一次用户输入的语义匹配某一条意图的**意图名称**和**意图说明**，则将这条意图的**意图id**输出为 `intention_id`\n- 禁止自行创建或修改 `intention_id`，其值必须严格来自两个列表中的意图id，或为 \'others\'\n- 选择优先级说明：\n  - 如果两个列表均有内容，首先使用【意图库列表】判别用户的意图，当匹配成功，将**意图id**输出为 `intention_id`；若无法匹配，才使用【知识库列表】判别用户的意图，当匹配成功

In [25]:
tokenizer.decode(list(filter(lambda x:x!=-100,tokenized_ds["train"][2]["labels"])))

'{"input_summary": "用户否认报名", "intention_id": "7b2e9f1c3a4d8e60"}<|im_end|>\n<|im_end|>'

In [26]:
tokenized_ds["train"][2]["input_ids"]

[151644,
 872,
 198,
 565,
 2049,
 220,
 103929,
 100780,
 53481,
 33108,
 102193,
 27369,
 9909,
 110079,
 7552,
 2049,
 198,
 105043,
 102192,
 106668,
 9370,
 105041,
 3837,
 85106,
 72064,
 104235,
 20002,
 81167,
 64471,
 110148,
 90395,
 99553,
 105674,
 89656,
 3407,
 565,
 2049,
 220,
 103929,
 100185,
 88802,
 9909,
 100645,
 105389,
 7552,
 2049,
 198,
 112735,
 100345,
 87752,
 102857,
 104317,
 334,
 117241,
 20002,
 31196,
 334,
 9370,
 111450,
 3837,
 101889,
 334,
 99373,
 66017,
 46944,
 5370,
 64429,
 334,
 3837,
 101244,
 53153,
 20412,
 92894,
 99885,
 68805,
 271,
 14374,
 220,
 99335,
 104190,
 28311,
 16,
 13,
 220,
 56568,
 91680,
 100668,
 104317,
 111450,
 3837,
 334,
 16530,
 71817,
 105051,
 1019,
 17,
 13,
 220,
 100783,
 105051,
 100022,
 100007,
 3837,
 103929,
 334,
 66017,
 100645,
 20412,
 100136,
 99373,
 101909,
 5370,
 64429,
 334,
 3837,
 99925,
 66017,
 50647,
 5373,
 99925,
 66017,
 92894,
 68805,
 105918,
 198,
 18,
 13,
 10236,
 99,
 223,
 81433

In [27]:
tokenized_ds["train"][2]["labels"]

[-100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,

In [28]:
len(tokenized_ds["train"][2]["labels"])

1327

## 3. Create model

Load the base model: Qwen3-4B-Instruct.

In [29]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="flash_attention_2"
)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [30]:
model.device

device(type='cuda', index=0)

In [31]:
# Disable cache during training to save memory, prediction in training is in parallel, not one by one.
model.config.use_cache = False

# Enable gradient checkpointing for memory efficiency
model.gradient_checkpointing_enable()

In [32]:
model = prepare_model_for_kbit_training(model)

In [33]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

In [34]:
sum(param.numel() for param in model.parameters())

4022468096

## 4. LoRA Configuration

### 4.1 PEFT step 1, LoRA adapter configuration

For the 4B model, we use r=32, and alpha=64.

In [35]:
config = LoraConfig(
    task_type = TaskType.CAUSAL_LM,
    r = 32,
    lora_alpha = 64,
    lora_dropout = 0.05,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none"
)

In [36]:
config

LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=32, target_modules={'q_proj', 'gate_proj', 'o_proj', 'v_proj', 'down_proj', 'up_proj', 'k_proj'}, exclude_modules=None, lora_alpha=64, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, arrow_config=None, ensure_weight_tying=False)

### 4.2 PEFT step 2, attach the LoRA adapter to the model

In [37]:
model = get_peft_model(model, config)

In [38]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [39]:
model.print_trainable_parameters()

trainable params: 66,060,288 || all params: 4,088,528,384 || trainable%: 1.6157


The parameters that need to be adjusts is significantly decreased from 4B to 66M.

In [40]:
for name, parameter in model.named_parameters():
    print(name)

base_model.model.model.embed_tokens.weight
base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.k_proj.base_layer.weight
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.v_proj.base_layer.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.o_proj.base_layer.weight
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.q_norm.weight
base_model.model.model.layers.0.self_attn.k_norm.weight
base_mode

## 5. Configure the training

In [41]:
args = TrainingArguments(
    output_dir=OUTPUT_DIR, #to store the prediction results and checkpoints of the model file
    per_device_train_batch_size = 2, #8 by default
    per_device_eval_batch_size = 2,
    gradient_accumulation_steps = 4, # Effective batch size = 8

    num_train_epochs=3,

    learning_rate=1e-4,
    lr_scheduler_type="cosine",

    # ~8000 training samples
    warmup_ratio=0.03,
    weight_decay=0.01,
    
    max_grad_norm=1.0,
    
    # Logging
    logging_steps=10,
    logging_first_step=True,

    # Evaluation
    eval_strategy="steps",
    eval_steps=150, #~8000 training samples

    # Saving
    save_strategy="steps",
    save_steps=150,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Precision
    bf16=True,
    tf32=True,
    
    # Memory optimization
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # group_by_length=True,
    
    # Performance
    dataloader_num_workers=4,
    report_to="none",
    remove_unused_columns=True # Set to True since we cleaned the dataset columns
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [42]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100
)

## 6. Create the trainer and train

In [43]:
trainer = Trainer(
    model = model,
    args = args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator = data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [44]:
trainer.train()

Casting fp32 inputs back to torch.bfloat16 for flash-attn compatibility.


Step,Training Loss,Validation Loss
150,0.184549,0.142575
300,0.104039,0.127554
450,0.133910,0.120822
600,0.103448,0.119755
750,0.125797,0.117165
900,0.128791,0.114452
1050,0.108794,0.111933
1200,0.082346,0.114915
1350,0.083645,0.112716
1500,0.092389,0.112492


TrainOutput(global_step=1500, training_loss=0.13116677125295004, metrics={'train_runtime': 15193.5157, 'train_samples_per_second': 1.496, 'train_steps_per_second': 0.187, 'total_flos': 4.1347105816028774e+17, 'train_loss': 0.13116677125295004, 'epoch': 1.5827395091053047})

## 7. Save model

In [45]:
# Create output directory with timestamp
adapter_dir = f"./lora_adapter"

In [46]:
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

print("Model saved successfully!")

Model saved successfully!
